In [ ]:
def load_vector_outputs(path: Path):
    with path.open() as f:
        data = json.load(f)
    return data.get("vector_outputs", {}), data


def extract_shares(vector_outputs: dict):
    ftg = pd.Series(
        vector_outputs.get("saf_ftg_mandate_share", []),
        index=range(2000, 2000 + len(vector_outputs.get("saf_ftg_mandate_share", []))),
    )
    co2 = pd.Series(
        vector_outputs.get("saf_co2_mandate_share", []),
        index=range(2000, 2000 + len(vector_outputs.get("saf_co2_mandate_share", []))),
    )
    lcaf = pd.Series(
        vector_outputs.get("lcaf_mandate_share", []),
        index=range(2000, 2000 + len(vector_outputs.get("lcaf_mandate_share", []))),
    )

    years = list(range(2000, 2000 + len(ftg)))
    idx = pd.Index(years, name="year")

    df = pd.DataFrame(
        {
            "saf_ftg": ftg,
            "saf_co2": co2,
            "lcaf": lcaf,
        },
        index=idx,
    )

    print(ftg, df)

    df = df.apply(pd.to_numeric, errors="coerce")
    # Fill NaN leading gaps forward/backward for plotting continuity
    df = df.fillna(method="ffill").fillna(0)
    df["kerosene"] = (100 - df[["saf_ftg", "saf_co2", "lcaf"]].sum(axis=1)).clip(lower=0)
    return df


def plot_blend(df: pd.DataFrame, title: str):
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.stackplot(
        df.index,
        [df["kerosene"], df["lcaf"], df["saf_co2"], df["saf_ftg"]],
        labels=["Kerosene", "LCAF", "SAF CO2", "SAF FTG"],
        colors=["#7f7f7f", "#ff7f0e", "#1f77b4", "#2ca02c"],
        alpha=0.85,
    )
    ax.set_ylabel("Blend share [%]")
    ax.set_xlabel("Year")
    ax.set_title(title)
    ax.set_ylim(0, 110)
    ax.legend(loc="upper right")
    ax.grid(True, alpha=0.3)
    return fig, ax

In [ ]:
df